# **IRIS Agents**

In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')

iris_toolkit

Toolkit(name='IRIS', url='http://localhost:9002/mcp')

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name='Agent007', text='You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name='Agent007', text='Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt('Agent007')
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production(name='AgentSpace', agents=[molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 05/11/2026 16:26:46
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 05/11/2026 16:26:46
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 05/11/2026 16:26:47
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 05/11/2026 16:26:47
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 05/11/2026 16:26:47
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are good summer hiking options near Boston (most within 45–60 minutes):\n- Blue Hills Reservation (Milton/Canton): Skyline Trail up to ~9 miles or shorter loops to Great Blue Hill; rocky, big skyline views.\n- Middlesex Fells (Medford/Winchester): Skyline 7–8 miles or Spot Pond/Reservoir loops; many shaded trails; Orange Line to Oak Grove helps.\n- Lynn Woods Reservation (Lynn): Stone Tower and Dungeon Rock; 2–8+ mile networks; moderate; commuter rail to Lynn plus bus.\n- Noanet Woodlands (Dover): 3–6 miles; Noanet Peak overlook of Boston; Trustees property.\n- Rocky Woods (Medfield): Easy family-friendly pond loops; shaded; Trustees.\n- Walden Pond and Walden Woods (Concord): 1.7-mile pond loop with swim options and longer forest trails; parking fills early.\n- Minute Man NHP Battle Road (Lexington–Concord): Flat 5-mile one-way path; historic, mostly shaded.\n- Great Meadows NWR (Concord): 1–3 mile dike trails; great birding; bring bug spray.\n- World’s End (Hingham): 4–5 miles 

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

Agents can be configured with a default reasoning effort and response format, but these parameters can be overridden at runtime.

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             reasoning_effort='low',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 05/11/2026 16:27:18
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 05/11/2026 16:27:19
Loading file Agents.Message.MollyResponse.cls as udl
Compiling class Agents.Message.MollyResponse
Compiling table Agents_Message.MollyResponse
Compiling routine Agents.Message.MollyResponse.1
Load finished successfully.


In [15]:
Production(name='AgentSpace', agents=[molly, alex]).start()


Load started on 05/11/2026 16:27:21
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 05/11/2026 16:27:21
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 05/11/2026 16:27:21
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 05/11/2026 16:27:21
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 05/11/2026 16:27:22
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

Without any context, the agent does not construct its memory from the database, nor does it log the response, operating in a stateless manner. Model usage is still logged.

In [16]:
molly('Which tables do we have in IRIS in the Agents namespace?')

'{"text": "Tables in Agents: SQLUser.TestModel", "reasoning": "Retrieved the list via IRIS list_tables for namespace \'Agents\'."}'

Agents can be directly invoked to use a tool using compatible parameters

In [17]:
molly.use(toolkit='IRIS', tool='query', params={'namespace':'Agents', 'sql': 'Select * from Agents.Agent'})

[{'agent_name': 'Alex',
  'description': 'Test Agent 1',
  'system_prompt_id': 'alex_system',
  'model': 'gpt-5',
  'response_format': 'AlexResponse',
  'pydantic_schema': '{"properties": {"message": {"title": "Message", "type": "string"}, "reasoning": {"title": "Reasoning", "type": "string"}}, "required": ["message", "reasoning"], "title": "AlexResponse", "type": "object"}',
  'reasoning_effort': 'medium',
  'persist_reasoning': '1'},
 {'agent_name': 'Molly',
  'description': 'Test Agent 2',
  'system_prompt_id': 'molly_system',
  'model': 'gpt-5',
  'response_format': 'MollyResponse',
  'pydantic_schema': '',
  'reasoning_effort': 'low',
  'persist_reasoning': '1'}]

When context is specified, the agent constructs its conversational context from database before querying the LLM

In [18]:
molly(message='What is the weather today?', chat=context)

'{"text": "Washington DC weather today: Cloudy with a high of 26\\u00b0 and a low of 13\\u00b0.", "reasoning": "Used the Utilities.weather tool result provided, which included the city, high, low, and sky conditions for Washington DC."}'

Here, we override the agent's default reasoning effort at runtime

In [19]:
molly(message='Recommend some good food spots for lunch', chat='travel', reasoning_effort='high')

'{"text": "Here are solid DC lunch picks across styles and neighborhoods:\\n- Le Diplomate (Logan Circle) \\u2013 French brasserie; steak frites, oysters.\\n- Old Ebbitt Grill (Downtown, near the White House) \\u2013 classic DC; crab cakes, raw bar.\\n- Unconventional Diner (Mt Vernon Sq) \\u2013 modern comfort food; salads, fried chicken.\\n- Jaleo (Penn Quarter) \\u2013 Spanish tapas by Jos\\u00e9 Andr\\u00e9s.\\n- Zaytinya (Penn Quarter) \\u2013 Mediterranean mezze; great for sharing.\\n- Shouk (multiple locations) \\u2013 plant-based Middle Eastern; pitas and bowls.\\n- RASA (Navy Yard + others) \\u2013 customizable Indian bowls; very veg-friendly.\\n- Indigo (NoMa/H St NE) \\u2013 homestyle Indian; thalis and biryani.\\n- Ben\\u2019s Chili Bowl (U Street) \\u2013 iconic half-smokes and chili.\\n- Union Market (NoMa) \\u2013 food hall with many vendors; easy for groups.\\n- Stachowski\\u2019s Market (Georgetown) \\u2013 excellent deli sandwiches.\\n- Wiseguy Pizza (multiple locatio

Here we override the agent's default response format at runtime

In [20]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 05/11/2026 16:28:52
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 05/11/2026 16:28:52
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "Fiola Mare", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "L\\u2019Ardente", "cuisine": "Italian"}, {"name": "Il Canale", "cuisine": "Italian"}, {"name": "2Amys", "cuisine": "Italian (Pizza)"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "Officina", "cuisine": "Italian"}, {"name": "Stellina Pizzeria", "cuisine": "Italian (Pizza)"}, {"name": "RPM Italian", "cuisine": "Italian"}, {"name": "Filomena Ristorante", "cuisine": "Italian"}, {"name": "Daikaya", "cuisine": "Japanese (Ramen)"}, {"name": "Haikan", "cuisine": "Japanese (Ramen)"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Thip Khao", "cuisine": "Lao"}, {"name": "Sushi Taro", "cuisine": "Japanese (Sushi)"}, {"name": "Sushi Capitol", "cuisine": "Japanese (Sushi)"}, {"name": "Tiger Fork", "cuisine": "Hong Kong"}, {"name": "Maketto", "cuisine": "Cambodian/Taiwanese"}, {"name": "O-Ku DC", "cuisine": "Japanese (Sushi)"}, {"name": "Panda Gourmet", "cuisine":

Usage information can be fetched using the API for each Agent, Production or Chat. Productions can also be filtered by certain agents.

In [21]:
Chat('travel').usage()

{'input_tokens': 3333,
 'output_tokens': 6970,
 'output_reasoning_tokens': 6080,
 'total_tokens': 10303}

In [22]:
Production('AgentSpace').usage()

{'input_tokens': 4960,
 'output_tokens': 9479,
 'output_reasoning_tokens': 8000,
 'total_tokens': 14439}

In [23]:
molly.usage()

{'input_tokens': 4960,
 'output_tokens': 9479,
 'output_reasoning_tokens': 8000,
 'total_tokens': 14439}

In [24]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 4960,
 'output_tokens': 9479,
 'output_reasoning_tokens': 8000,
 'total_tokens': 14439}

Productions and Agents can be deleted using the `delete()` methods. This operation removes the Objectscript classes backing these objects as well.

In [25]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [26]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
